# PS1 3-day TVM — Improved Baseline (models only)

Focused modeling notebook: **no MLflow, no SageMaker deploy**. Optional **Optuna HPT** in Cell 9b.

**Silver join notes (catalog):**
| Table | Join | TVM note |
|---|---|---|
| `maintenance_ledger`, `device_outage`, `device_ps2_chains` | `DEVICE_KEY` + `event_date` | all categories |
| `read_tap_daily` | `DEVICE_ID` + `event_date` | **often ~0% match** in current export — null-prune drops dead cols |
| `station_network_daily` | `FACILITY_ID` + `TVM` + `event_date` | ~7% match (sparse failure-day rows) |
| `usage_lifecycle_daily`, `metric_daily` | `DEVICE_KEY` + `event_date` | usage ~0% for TVM (VALIDATOR/GATE SQL scope); metric tried, pruned if dead |

**Modeling choices (TVM v2 — prod-aligned fixes):**
- **Natural `scale_pos_weight`** (~5.4) — no SPW sweep (sweep collapsed LGB at 15% pos rate)
- **Recency weighting** (180-day half-life) — addresses monthly AUC drift Jan→Apr
- **No oversample** — prod trains on raw train
- Val AP-weighted LGB/XGB/CB/RF ensemble; isotonic **only if val AP improves**
- **Optional Optuna HPT** (Cell 9b): TPE on LGB/XGB/CB — set `USE_OPTUNA=True`
- MI keep top **120**; force-protect sales/outage/kpi/subsystem SHAP drivers
- LGB early-stop params match prod (`n_estimators=600`, `reg_lambda=1.0`)

In [ ]:
# CELL 1 — Install
import subprocess, sys
pkgs = ["s3fs", "pyarrow", "scikit-learn>=1.4", "imbalanced-learn>=0.12",
        "xgboost>=2.0", "lightgbm>=4.3", "catboost>=1.2", "pandas>=2.0",
        "matplotlib", "seaborn", "optuna>=3.0"]
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + pkgs)
print("Packages OK.")

In [ ]:
# CELL 2 — Imports, config, join helpers
import os, warnings
import decimal as _decimal
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.feature_selection import mutual_info_classif
from sklearn.metrics import (
    roc_auc_score, average_precision_score, precision_recall_curve,
    f1_score, precision_score, recall_score, roc_curve, confusion_matrix,
)
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier, early_stopping, log_evaluation
from catboost import CatBoostClassifier
from sklearn.isotonic import IsotonicRegression

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 60)

GOLD_BUCKET = "cubic-mars-pm-s3-datalake-dev-gold-170202974600"
GOLD_S3     = f"s3://{GOLD_BUCKET}/chicago/gold/device_ps1_daily"
SILVER      = f"s3://{GOLD_BUCKET}/chicago/silver"
REGION      = "us-east-1"
S3_OPTS     = {"client_kwargs": {"region_name": REGION}}

DEVICE_CATEGORY  = "TVM"
TARGET           = "will_fail_3d"
VAL_START        = pd.Timestamp("2025-10-01")
TEST_START       = pd.Timestamp("2026-01-01")
EMBARGO_DAYS     = 14
RECALL_FLOOR     = 0.80
NULL_DROP_THRESH = 0.90
ZERO_DROP_THRESH = 0.95
USE_MI_PRUNING   = True
MI_TOP_N         = 120
LEAKAGE_AUC_TH   = 0.95
USE_SPW_SWEEP    = False              # TVM ~15% pos: sweep over-weights LGB; use natural ratio
SPW_CANDIDATES   = [4, 5, 6, 7, 8]    # only used if USE_SPW_SWEEP=True
RECENCY_HALF_LIFE_DAYS = 180          # prod TVM drift mitigation (monthly AUC decay)
USE_ISOTONIC     = True               # guarded: skip if val AP drops
MIN_BOOSTER_VAL_AP = 0.15             # drop collapsed models from ensemble
USE_OVERSAMPLE   = False
OVERSAMPLE_RATIO = 0.05
SIGNAL_PROTECT_PAT = (
    "sales", "revenue", "kpi", "scrst", "system_events", "csc_reader", "roll_fail",
    "outage", "availability", "device_age", "events_7d", "oos_events",
    "reject_rate", "null_rate", "bhu", "chu", "alarm", "comms",
)
OUT_DIR = "ps1_tvm_improved_outputs"
os.makedirs(OUT_DIR, exist_ok=True)
# --- Optional Optuna HPT (Cell 9b) — prod-aligned TPE on train+val TimeSeriesSplit ---
USE_OPTUNA         = False            # True: tune LGB/XGB/CB in Cell 9b; skip default Cells 10-12
FORCE_OPTUNA       = False            # ignore checkpoint and re-run all studies
OPTUNA_CKPT        = os.path.join(OUT_DIR, "optuna_params_tvm.joblib")
OPTUNA_TUNE_ROWS   = 400_000          # row cap for search only (final refit uses full train)
OPTUNA_TRIALS      = {"xgb": 40, "lgb": 30, "cb": 20}
OPTUNA_TIMEOUT_SEC = {"xgb": 600, "lgb": 480, "cb": 360}
_MODEL_TAG         = "improved"       # overwritten to "optuna" when USE_OPTUNA=True


def _cast_decimals(frame):
    dec_cols = [c for c in frame.columns if frame[c].dtype == object
                and len(frame[c].dropna()) > 0
                and isinstance(frame[c].dropna().iloc[0], _decimal.Decimal)]
    if dec_cols:
        frame[dec_cols] = frame[dec_cols].apply(pd.to_numeric, errors="coerce")
    return frame


def _norm_facility(s):
    return pd.to_numeric(s, errors="coerce")


def _report_signal(df, col, label):
    if col not in df.columns:
        print(f"  [{label}] {col}: missing")
        return 0.0
    nn = df[col].notna().mean()
    nz = (df[col].fillna(0) != 0).mean()
    print(f"  [{label}] {col}: notna={nn:.1%}  non-zero={nz:.1%}")
    return nn


def _prior_roll(g, col, window, how="mean"):
    if how == "mean":
        return g[col].transform(lambda s: s.shift(1).rolling(window, min_periods=1).mean())
    if how == "sum":
        return g[col].transform(lambda s: s.shift(1).rolling(window, min_periods=1).sum())
    if how == "max":
        return g[col].transform(lambda s: s.shift(1).rolling(window, min_periods=1).max())
    raise ValueError(how)

print(f"Gold: {GOLD_S3}")

In [ ]:
# CELL 3 — Load Gold, TVM filter, column aliases
df = pd.read_parquet(GOLD_S3, storage_options=S3_OPTS)
if "transit_day" in df.columns and "event_date" not in df.columns:
    df = df.rename(columns={"transit_day": "event_date"})
df["event_date"] = pd.to_datetime(df["event_date"])
df = df[df["mars_device_category"] == DEVICE_CATEGORY].copy()
df = _cast_decimals(df)
df = df.sort_values(["DEVICE_ID", "event_date"]).reset_index(drop=True)

if "event_count" in df.columns and "daily_event_count" not in df.columns:
    df = df.rename(columns={"event_count": "daily_event_count"})
if "oos_event_count" in df.columns and "oos_flag" not in df.columns:
    df["oos_flag"] = (df["oos_event_count"] > 0).astype(int)

print(f"TVM rows: {len(df):,}  devices: {df['DEVICE_ID'].nunique():,}")
print(f"DEVICE_KEY coverage: {df['DEVICE_KEY'].notna().mean():.1%}")
print(f"Date range: {df['event_date'].min().date()} -> {df['event_date'].max().date()}")
print(f"Target rate: {df[TARGET].mean():.2%}")

In [ ]:
# CELL 4 — Silver enrichments (schema-correct joins, TVM-specific)

def _g():
    return df.groupby("DEVICE_ID", sort=False)

print("Silver enrichment join audit:")

# --- maintenance_ledger: DEVICE_KEY + event_date ---
try:
    maint = pd.read_parquet(f"{SILVER}/maintenance_ledger", storage_options=S3_OPTS,
                            columns=["DEVICE_KEY", "event_dtm", "ledger_type"])
    maint = maint[maint["ledger_type"] != "REPAIR_EPISODE"].copy()
    maint["event_date"] = pd.to_datetime(maint["event_dtm"]).dt.normalize()
    md = maint.groupby(["DEVICE_KEY", "event_date"]).size().reset_index(name="_maint_flag")
    md["_maint_flag"] = 1
    n0 = len(df)
    df = df.merge(md, on=["DEVICE_KEY", "event_date"], how="left", validate="m:1")
    assert len(df) == n0
    df["_maint_flag"] = df["_maint_flag"].fillna(0).astype(int)
    df = df.sort_values(["DEVICE_ID", "event_date"])
    g = _g()
    df["_maint_lag"] = g["_maint_flag"].shift(1).fillna(0)
    df["maint_visit_days_7d"]  = g["_maint_lag"].transform(lambda x: x.rolling(7, min_periods=1).sum())
    df["maint_visit_days_30d"] = g["_maint_lag"].transform(lambda x: x.rolling(30, min_periods=1).sum())
    df["_last_maint"] = df["event_date"].where(df["_maint_lag"] == 1)
    df["_last_maint"] = g["_last_maint"].ffill()
    df["days_since_maintenance"] = (df["event_date"] - df["_last_maint"]).dt.days.fillna(999).clip(upper=999)
    df = df.drop(columns=["_maint_flag", "_maint_lag", "_last_maint"])
    _report_signal(df, "days_since_maintenance", "MAINT")
except Exception as e:
    print(f"  [MAINT] skip: {e}")

# --- device_ps2_chains ---
try:
    ps2 = pd.read_parquet(f"s3://{GOLD_BUCKET}/chicago/gold/device_ps2_chains", storage_options=S3_OPTS,
                          columns=["DEVICE_KEY", "transit_day", "is_coordinated_station_failure"])
    ps2["event_date"] = pd.to_datetime(ps2["transit_day"])
    p2 = ps2.groupby(["DEVICE_KEY", "event_date"]).agg(
        _coord=("is_coordinated_station_failure", "max")).reset_index()
    p2["_cascade_flag"] = 1
    p2["_coord_flag"] = p2["_coord"].astype(int)
    n0 = len(df)
    df = df.merge(p2[["DEVICE_KEY", "event_date", "_cascade_flag", "_coord_flag"]],
                  on=["DEVICE_KEY", "event_date"], how="left", validate="m:1")
    assert len(df) == n0
    for c in ["_cascade_flag", "_coord_flag"]:
        df[c] = df[c].fillna(0).astype(int)
    df = df.sort_values(["DEVICE_ID", "event_date"])
    g = _g()
    df["_casc_lag"] = g["_cascade_flag"].shift(1).fillna(0)
    df["_coord_lag"] = g["_coord_flag"].shift(1).fillna(0)
    df["station_cascade_days_30d"] = g["_casc_lag"].transform(lambda x: x.rolling(30, min_periods=1).sum())
    df["station_coordinated_fail_days_30d"] = g["_coord_lag"].transform(lambda x: x.rolling(30, min_periods=1).sum())
    df["_last_casc"] = df["event_date"].where(df["_casc_lag"] == 1)
    df["_last_casc"] = g["_last_casc"].ffill()
    df["days_since_station_cascade"] = (df["event_date"] - df["_last_casc"]).dt.days.fillna(999).clip(upper=999)
    df = df.drop(columns=["_cascade_flag", "_coord_flag", "_casc_lag", "_coord_lag", "_last_casc"])
    _report_signal(df, "days_since_station_cascade", "PS2")
except Exception as e:
    print(f"  [PS2] skip: {e}")

# --- device_outage: 8 TVM subsystem buckets ---
try:
    _SUBSYSTEMS = ["SYSTEM", "CSC_READER", "SCRST", "BHU", "CHU", "ALARM", "COMMS", "OTHER"]
    _out = pd.read_parquet(f"{SILVER}/device_outage", storage_options=S3_OPTS,
                           columns=["DEVICE_KEY", "transit_day", "mars_device_category", "duration_min",
                                    "is_chargeable", "failure_level", "component_subsystem"])
    _out = _out[(_out["mars_device_category"] == "TVM") & (_out["duration_min"] > 0)
                & (_out["is_chargeable"] == True)
                & (_out["failure_level"].isin([2, 3, 4, 5]))].copy()
    _out["event_date"] = pd.to_datetime(_out["transit_day"])
    _known = set(_SUBSYSTEMS[:-1])
    _out["_sub"] = np.where(_out["component_subsystem"].isin(_known), _out["component_subsystem"], "OTHER")
    _out_daily = _out.groupby(["DEVICE_KEY", "event_date", "_sub"]).size().unstack("_sub", fill_value=0).reset_index()
    for _s in _SUBSYSTEMS:
        if _s not in _out_daily.columns:
            _out_daily[_s] = 0
        _out_daily[_s] = (_out_daily[_s] > 0).astype(int)
    n0 = len(df)
    df = df.merge(_out_daily, on=["DEVICE_KEY", "event_date"], how="left", validate="m:1")
    assert len(df) == n0
    df = df.sort_values(["DEVICE_ID", "event_date"])
    g = _g()
    for _s in _SUBSYSTEMS:
        df[_s] = df[_s].fillna(0).astype(int)
        _lagcol = f"_lag_{_s}"
        df[_lagcol] = g[_s].shift(1).fillna(0)
        df[f"roll_fail_{_s}_30d"] = g[_lagcol].transform(lambda x: x.rolling(30, min_periods=1).sum())
        _lastcol = f"_last_{_s}"
        df[_lastcol] = df["event_date"].where(df[_lagcol] == 1)
        df[_lastcol] = g[_lastcol].ffill()
        df[f"days_since_fail_{_s}"] = (
            (df["event_date"] - df[_lastcol]).dt.days.fillna(999).clip(upper=999))
    _temp = _SUBSYSTEMS + [f"_lag_{s}" for s in _SUBSYSTEMS] + [f"_last_{s}" for s in _SUBSYSTEMS]
    df = df.drop(columns=[c for c in _temp if c in df.columns])
    _report_signal(df, "roll_fail_SCRST_30d", "SUBSYS SCRST")
    _report_signal(df, "days_since_fail_CSC_READER", "SUBSYS CSC_READER")
except Exception as e:
    print(f"  [SUBSYS] skip: {e}")

# --- read_tap_daily: DEVICE_ID + event_date (TVM-specific; often sparse) ---
try:
    rt = pd.read_parquet(f"{SILVER}/read_tap_daily", storage_options=S3_OPTS,
                         columns=["DEVICE_ID", "transit_day", "mars_device_category",
                                  "daily_read_count", "rejected_read_count", "null_status_read_count"])
    rt = rt[rt["mars_device_category"] == "TVM"].copy()
    rt["event_date"] = pd.to_datetime(rt["transit_day"])
    rt = rt.drop_duplicates(["DEVICE_ID", "event_date"])
    _denom = rt["daily_read_count"].where(rt["daily_read_count"] > 0)
    rt["reject_rate_pct"] = 100.0 * rt["rejected_read_count"] / _denom
    rt["null_rate_pct"]   = 100.0 * rt["null_status_read_count"] / _denom
    n0 = len(df)
    df = df.merge(rt[["DEVICE_ID", "event_date", "reject_rate_pct", "null_rate_pct"]],
                  on=["DEVICE_ID", "event_date"], how="left", validate="m:1")
    assert len(df) == n0
    df = df.sort_values(["DEVICE_ID", "event_date"])
    g = _g()
    _report_signal(df, "reject_rate_pct", "READ_TAP raw")
    for c in ["reject_rate_pct", "null_rate_pct"]:
        df[f"{c}_7d_prior"] = g[c].transform(lambda s: s.shift(1).rolling(7, min_periods=1).mean())
        df = df.drop(columns=[c])
    _report_signal(df, "reject_rate_pct_7d_prior", "READ_TAP prior7d")
except Exception as e:
    print(f"  [READ_TAP] skip: {e}")

# --- station_network_daily ---
try:
    st = pd.read_parquet(f"{SILVER}/station_network_daily", storage_options=S3_OPTS,
                         columns=["FACILITY_ID", "device_category", "transit_day",
                                  "devices_failed", "total_failure_events",
                                  "is_coordinated_failure", "is_major_station_event"])
    st = st[st["device_category"] == "TVM"].copy()
    st["event_date"] = pd.to_datetime(st["transit_day"])
    st["FACILITY_ID"] = _norm_facility(st["FACILITY_ID"])
    df["_fac_join"] = _norm_facility(df["FACILITY_ID"])
    for c in ["is_coordinated_failure", "is_major_station_event"]:
        st[c] = st[c].astype(int)
    n0 = len(df)
    df = df.merge(st.drop(columns=["transit_day", "device_category"]).rename(columns={"FACILITY_ID": "_fac_join"}),
                  on=["_fac_join", "event_date"], how="left", validate="m:1")
    assert len(df) == n0
    df = df.sort_values(["DEVICE_ID", "event_date"])
    g = _g()
    _report_signal(df, "devices_failed", "STN_NET raw")
    for c, how, out in [
        ("devices_failed", "sum", "stn_devices_failed_7d_prior"),
        ("total_failure_events", "sum", "stn_failure_events_7d_prior"),
        ("is_coordinated_failure", "max", "stn_coordinated_7d_prior"),
        ("is_major_station_event", "max", "stn_major_7d_prior"),
    ]:
        df[c] = df[c].fillna(0.0)
        df[out] = _prior_roll(g, c, 7, how=how)
        df = df.drop(columns=[c])
    df = df.drop(columns=["_fac_join"])
    _report_signal(df, "stn_devices_failed_7d_prior", "STN_NET prior7d")
except Exception as e:
    print(f"  [STN_NET] skip: {e}")

# --- usage_lifecycle_daily: try DEVICE_KEY (catalog-correct), expect low TVM coverage ---
try:
    ul = pd.read_parquet(f"{SILVER}/usage_lifecycle_daily", storage_options=S3_OPTS,
                         columns=["DEVICE_KEY", "transit_day", "mars_device_category", "m401_daily_txn_count"])
    ul = ul[ul["mars_device_category"] == "TVM"].copy()
    ul["event_date"] = pd.to_datetime(ul["transit_day"])
    ul = ul.drop_duplicates(["DEVICE_KEY", "event_date"])
    n0 = len(df)
    df = df.merge(ul.drop(columns=["transit_day", "mars_device_category"]),
                  on=["DEVICE_KEY", "event_date"], how="left", validate="m:1")
    assert len(df) == n0
    df = df.sort_values(["DEVICE_ID", "event_date"])
    g = _g()
    _report_signal(df, "m401_daily_txn_count", "USAGE raw (DEVICE_KEY)")
    df["m401_daily_txn_count"] = df["m401_daily_txn_count"].fillna(0)
    df["usage_tap_count_30d_prior"] = g["m401_daily_txn_count"].transform(
        lambda s: s.shift(1).rolling(30, min_periods=1).sum())
    df["usage_tap_count_lifetime_prior"] = g["m401_daily_txn_count"].transform(
        lambda s: s.shift(1).cumsum())
    df = df.drop(columns=["m401_daily_txn_count"])
except Exception as e:
    print(f"  [USAGE] skip: {e}")

# --- metric_daily: try DEVICE_KEY; null-prune will drop if dead on TVM ---
try:
    md = pd.read_parquet(f"{SILVER}/metric_daily", storage_options=S3_OPTS,
                         columns=["DEVICE_KEY", "transit_day", "mars_device_category",
                                  "m401_slow_tap_pct", "m401_z_score_vs_28d", "comms_total_count", "volume_drop_flag"])
    md = md[md["mars_device_category"] == "TVM"].copy()
    md["event_date"] = pd.to_datetime(md["transit_day"])
    md = md.drop_duplicates(["DEVICE_KEY", "event_date"])
    n0 = len(df)
    df = df.merge(md.drop(columns=["transit_day", "mars_device_category"]),
                  on=["DEVICE_KEY", "event_date"], how="left", validate="m:1")
    assert len(df) == n0
    df = df.sort_values(["DEVICE_ID", "event_date"])
    g = _g()
    _report_signal(df, "m401_slow_tap_pct", "METRIC_DAILY raw")
    for c in ["m401_slow_tap_pct", "m401_z_score_vs_28d", "comms_total_count"]:
        if c in df.columns:
            df[f"{c}_lag1"] = g[c].shift(1)
    if "volume_drop_flag" in df.columns:
        df["volume_drop_flag_lag1"] = g["volume_drop_flag"].astype(float).shift(1).fillna(0)
    df = df.drop(columns=["m401_slow_tap_pct", "m401_z_score_vs_28d",
                          "comms_total_count", "volume_drop_flag"], errors="ignore")
except Exception as e:
    print(f"  [METRIC_DAILY] skip: {e}")

print(f"\nShape after silver enrichments: {df.shape}")

In [ ]:
# CELL 5 — Notebook feature blocks A–L (strict shift(1), no same-day leakage)

ID_COLS = ["DEVICE_ID", "event_date", "split", "DEVICE_KEY", "FACILITY_ID",
           "OPERATOR_ID", "BUS_ID", "TRANSIT_ARRAY_ID", "ARRAY_POSITION", "mars_device_category"]
LABEL_LEAK = ["failure_level", "max_failure_level", "is_chargeable", "is_hardware_oos",
              "is_hardware_event", "duration_min", "oos_duration_min",
              "will_fail_7d", "will_fail_14d", TARGET]

g = df.groupby("DEVICE_ID", sort=False)

_GOLD_SKIP = {
    "oos_flag_roll7": "oos_events_7d",
    "oos_flag_roll30": "oos_events_30d",
    "daily_event_count_roll7": "events_7d",
}
for col, window in [("oos_flag", 7), ("oos_flag", 30), ("daily_event_count", 7)]:
    nb = f"{col}_roll{window}"
    if col in df.columns and nb not in _GOLD_SKIP:
        df[nb] = g[col].transform(lambda x: x.shift(1).rolling(window, min_periods=1).sum())
    elif col in df.columns and _GOLD_SKIP.get(nb) in df.columns:
        print(f"  [FE] skip {nb} — gold has {_GOLD_SKIP[nb]}")

if "daily_event_count" in df.columns:
    df["event_rate_zscore"] = g["daily_event_count"].transform(
        lambda x: (x.shift(1) - x.shift(1).expanding().mean()) / (x.shift(1).expanding().std() + 1e-6))

if "use_revenue_cents" in df.columns:
    df["use_revenue_cents_roll7"] = g["use_revenue_cents"].transform(
        lambda x: x.shift(1).rolling(7, min_periods=1).mean())

for col in [c for c in ["oos_flag", "daily_event_count", "use_revenue_cents"] if c in df.columns]:
    for lag in [1, 2, 3]:
        df[f"{col}_lag{lag}"] = g[col].shift(lag)

if "oos_flag" in df.columns:
    for w in [30, 90]:
        df[f"oos_count_{w}d"] = g["oos_flag"].transform(lambda x, w=w: x.shift(1).rolling(w, min_periods=1).sum())
    def _days_since_oos(x):
        s = x.shift(1); cs = s.cumsum()
        out = cs.groupby(cs).cumcount().astype(float); out[cs == 0] = np.nan; return out
    df["days_since_last_oos"] = g["oos_flag"].transform(_days_since_oos)
    df["oos_lifetime_count"] = g["oos_flag"].transform(lambda x: x.shift(1).cumsum())

df["device_age_days"] = g["event_date"].transform(lambda x: (x - x.min()).dt.days)

if "oos_flag" in df.columns:
    def _consec(x):
        s = x.shift(1).fillna(0); grp = (s != s.shift(1)).cumsum()
        streak = s.groupby(grp).cumcount() + 1; streak[s == 0] = 0.0; return streak.astype(float)
    df["consec_oos_days"] = g["oos_flag"].transform(_consec)
    df["max_streak_30d"] = g["consec_oos_days"].transform(lambda x: x.shift(1).rolling(30, min_periods=1).max())
    r7  = g["oos_flag"].transform(lambda x: x.shift(1).rolling(7, min_periods=1).sum())
    p7  = g["oos_flag"].transform(lambda x: x.shift(8).rolling(7, min_periods=1).sum())
    r30 = g["oos_flag"].transform(lambda x: x.shift(1).rolling(30, min_periods=1).sum())
    p30 = g["oos_flag"].transform(lambda x: x.shift(31).rolling(30, min_periods=1).sum())
    df["oos_rate_accel_7d"]  = (r7 - p7).fillna(0)
    df["oos_rate_accel_30d"] = (r30 - p30).fillna(0)
    df["oos_ewm7"]  = g["oos_flag"].transform(lambda x: x.shift(1).ewm(span=7, min_periods=1).mean())
    df["oos_ewm30"] = g["oos_flag"].transform(lambda x: x.shift(1).ewm(span=30, min_periods=1).mean())
    l1 = g["oos_flag"].shift(1).fillna(0); l2 = g["oos_flag"].shift(2).fillna(0)
    df["_recovery"] = ((l2 == 1) & (l1 == 0)).astype(float)
    df["recovery_count_30d"] = g["_recovery"].transform(lambda x: x.rolling(30, min_periods=1).sum())
    df["recovery_count_90d"] = g["_recovery"].transform(lambda x: x.rolling(90, min_periods=1).sum())
    df = df.drop(columns=["_recovery"])

if "daily_event_count" in df.columns:
    mu  = g["daily_event_count"].transform(lambda x: x.shift(1).rolling(30, min_periods=5).mean())
    std = g["daily_event_count"].transform(lambda x: x.shift(1).rolling(30, min_periods=5).std())
    df["event_count_zscore_30d"] = ((g["daily_event_count"].shift(1) - mu) / (std + 1e-6)).fillna(0)

if "use_revenue_cents" in df.columns:
    mu  = g["use_revenue_cents"].transform(lambda x: x.shift(1).rolling(30, min_periods=5).mean())
    std = g["use_revenue_cents"].transform(lambda x: x.shift(1).rolling(30, min_periods=5).std())
    df["revenue_zscore_30d"] = ((g["use_revenue_cents"].shift(1).astype(float) - mu) / (std + 1e-6)).fillna(0)

df["day_of_week"]  = df["event_date"].dt.dayofweek
df["month"]        = df["event_date"].dt.month
df["day_of_month"] = df["event_date"].dt.day
df["is_weekend"]   = (df["day_of_week"] >= 5).astype(int)
df["quarter"]      = df["event_date"].dt.quarter

if "FACILITY_ID" in df.columns and "oos_flag" in df.columns:
    df["_oos_lag1"] = g["oos_flag"].shift(1).fillna(0)
    fac_mean = df.groupby(["event_date", "FACILITY_ID"])["_oos_lag1"].transform("mean")
    df["peer_oos_rate_lag1"] = fac_mean
    df["oos_vs_peer_lag1"]   = df["_oos_lag1"] - fac_mean
    df = df.drop(columns=["_oos_lag1"])

if "oos_ewm7" in df.columns:
    df["oos_ewm7_x_age"] = df["oos_ewm7"] * np.log1p(df["device_age_days"])
if "oos_flag" in df.columns and "daily_event_count" in df.columns:
    df["oos_x_event_count"] = g["oos_flag"].shift(1).fillna(0) * g["daily_event_count"].shift(1).fillna(0)

for dc in ["availability_pct_7d", "outage_min_7d"]:
    if dc in df.columns:
        df[dc] = pd.to_numeric(df[dc], errors="coerce").astype(float)

# TVM SHAP drivers: lagged sales / subsystem event intensity
if "sales_7d_avg" in df.columns:
    df["sales_7d_avg_lag1"] = g["sales_7d_avg"].shift(1)
    df["sales_7d_avg_roll7_prior"] = g["sales_7d_avg"].transform(
        lambda x: x.shift(1).rolling(7, min_periods=1).mean())
    _report_signal(df, "sales_7d_avg_lag1", "TVM sales lag1")
for _evcol in ["scrst_events", "system_events", "csc_reader_events", "comms_events",
                "bhu_events", "chu_events", "kpi_count"]:
    if _evcol in df.columns:
        df[f"{_evcol}_roll7_lag"] = g[_evcol].transform(
            lambda x: x.shift(1).rolling(7, min_periods=1).sum())

FEATURE_COLS = [c for c in df.columns if c not in ID_COLS + LABEL_LEAK and pd.api.types.is_numeric_dtype(df[c])]
print(f"Candidate features: {len(FEATURE_COLS)}")

In [ ]:
# CELL 6 — Leakage scan (solo-feature AUC on internal temporal holdout)
_scan_cut = df["event_date"].quantile(0.80)
_scan_tr  = df[df["event_date"] <= _scan_cut]
_scan_va  = df[df["event_date"] >  _scan_cut]

def _solo_auc(col):
    a = _scan_tr[col].astype(float); b = _scan_va[col].astype(float)
    med = a.median(); a = a.fillna(med); b = b.fillna(med)
    if a.nunique() < 2 or _scan_va[TARGET].nunique() < 2:
        return np.nan
    auc = roc_auc_score(_scan_va[TARGET], b)
    return max(auc, 1 - auc)

_scan = pd.Series({c: _solo_auc(c) for c in FEATURE_COLS}).sort_values(ascending=False)
_leaky = _scan[_scan > LEAKAGE_AUC_TH].index.tolist()
print("Top solo-AUC features:")
print(_scan.head(12).round(4).to_string())
if _leaky:
    print(f"\nDropping {len(_leaky)} leaky features: {_leaky}")
    FEATURE_COLS = [c for c in FEATURE_COLS if c not in _leaky]
else:
    print(f"\nNo feature exceeded solo-AUC {LEAKAGE_AUC_TH} — keeping {len(FEATURE_COLS)} features.")

In [ ]:
# CELL 7 — Split, dead-feature drop, impute, MI pruning, SPW sweep
tr = df[df["event_date"] <  VAL_START - pd.Timedelta(days=EMBARGO_DAYS)]
va = df[(df["event_date"] >= VAL_START) & (df["event_date"] < TEST_START - pd.Timedelta(days=EMBARGO_DAYS))]
te = df[df["event_date"] >= TEST_START]

null_rate = tr[FEATURE_COLS].isnull().mean()
dead = null_rate[null_rate > NULL_DROP_THRESH].index.tolist()
FEATURE_COLS = [c for c in FEATURE_COLS if c not in dead]
print(f"Dropped {len(dead)} dead features (>={NULL_DROP_THRESH:.0%} null on train)")
if dead:
    for c in dead:
        print(f"  {c:45s} null={null_rate[c]:.1%}")

X_tr_raw = tr[FEATURE_COLS].astype(float)
zero_rate = (X_tr_raw.fillna(0) == 0).mean()
sparse = zero_rate[zero_rate > ZERO_DROP_THRESH].index.tolist()
_SPARSE_KEEP = {c for c in sparse if any(k in c for k in (
    "roll_fail", "days_since_fail", "days_since_maintenance", "days_since_station",
    "stn_coordinated", "stn_major", "subsystem", "SYSTEM", "SCRST", "CSC_READER",
    "BHU", "CHU", "ALARM", "COMMS"))}
sparse = [c for c in sparse if c not in _SPARSE_KEEP]
FEATURE_COLS = [c for c in FEATURE_COLS if c not in sparse]
print(f"Dropped {len(sparse)} near-constant features (>{ZERO_DROP_THRESH:.0%} zero on train)")

X_tr = tr[FEATURE_COLS].astype(float)
X_va = va[FEATURE_COLS].astype(float)
X_te = te[FEATURE_COLS].astype(float)
y_tr, y_va, y_te = tr[TARGET].values, va[TARGET].values, te[TARGET].values

train_median = X_tr.median(numeric_only=True).values
train_median = np.where(np.isnan(train_median), 0.0, train_median)

def impute(X):
    arr = X.values.copy()
    mask = np.isnan(arr)
    arr[mask] = np.take(train_median, np.where(mask)[1])
    return arr

X_tr = impute(X_tr); X_va = impute(X_va); X_te = impute(X_te)

if USE_MI_PRUNING:
    mi = mutual_info_classif(X_tr, y_tr, random_state=42, discrete_features=False)
    mi_s = pd.Series(mi, index=FEATURE_COLS).sort_values(ascending=False)
    keep = mi_s.head(MI_TOP_N).index.tolist()
    _prot = [c for c in FEATURE_COLS if any(p in c.lower() for p in SIGNAL_PROTECT_PAT)]
    keep = list(dict.fromkeys(list(keep) + _prot))
    if _prot:
        print(f"  Signal-protected (+{len(_prot)}): {_prot[:6]}{'...' if len(_prot)>6 else ''}")
    print(f"\nMI pruning: {len(FEATURE_COLS)} -> {len(keep)}")
    print(mi_s.head(12).round(4).to_string())
    idx = [FEATURE_COLS.index(c) for c in keep]
    FEATURE_COLS = keep
    X_tr, X_va, X_te = X_tr[:, idx], X_va[:, idx], X_te[:, idx]

spw_raw = round((y_tr == 0).sum() / max((y_tr == 1).sum(), 1), 2)
print(f"\nTrain {X_tr.shape}  pos={(y_tr==1).sum():,} ({(y_tr==1).mean():.2%})")
print(f"Val   {X_va.shape}  pos={(y_va==1).sum():,} ({(y_va==1).mean():.2%})")
print(f"Test  {X_te.shape}  pos={(y_te==1).sum():,} ({(y_te==1).mean():.2%})")
print(f"scale_pos_weight raw={spw_raw}")

# Recency weights (prod TVM: down-weight rows >6mo old)
_decay = np.log(2) / RECENCY_HALF_LIFE_DAYS
_train_dates = tr["event_date"].values
_w_train = np.exp(-_decay * (tr["event_date"].max() - tr["event_date"]).dt.days.values)
_w_train = _w_train / _w_train.mean()
print(f"Recency weights: min={_w_train.min():.3f}  max={_w_train.max():.3f}  half_life={RECENCY_HALF_LIFE_DAYS}d")

if USE_SPW_SWEEP:
    _spw_scores = {}
    for _cand in SPW_CANDIDATES:
        _probe = LGBMClassifier(
            n_estimators=300, learning_rate=0.05, num_leaves=31, max_depth=6,
            scale_pos_weight=_cand, random_state=42, n_jobs=-1, verbose=-1,
        )
        _probe.fit(X_tr, y_tr, sample_weight=_w_train, eval_set=[(X_va, y_va)],
                   eval_metric="auc", callbacks=[early_stopping(30, verbose=False), log_evaluation(0)])
        _spw_scores[_cand] = average_precision_score(y_va, _probe.predict_proba(X_va)[:, 1])
    spw_fit = max(_spw_scores, key=_spw_scores.get)
    print(f"SPW sweep val AP: {_spw_scores}  -> spw_fit={spw_fit}")
else:
    spw_fit = spw_raw
    print(f"Using natural scale_pos_weight={spw_fit} (prod-aligned — no SPW sweep)")

if USE_OVERSAMPLE:
    from imblearn.over_sampling import RandomOverSampler
    _ros = RandomOverSampler(sampling_strategy=OVERSAMPLE_RATIO, random_state=42)
    X_tr_fit, y_tr_fit = _ros.fit_resample(X_tr, y_tr)
    _w_fit = np.ones(len(y_tr_fit))
    spw_fit = round((y_tr_fit == 0).sum() / max((y_tr_fit == 1).sum(), 1), 2)
    print(f"Oversample to {OVERSAMPLE_RATIO:.0%}: {X_tr.shape[0]:,} -> {X_tr_fit.shape[0]:,} rows")
else:
    X_tr_fit, y_tr_fit = X_tr, y_tr
    _w_fit = _w_train
    print(f"No oversample — spw_fit={spw_fit}")

In [ ]:
# CELL 8 — Metrics + ROC / confusion-matrix plots

_SPLIT_DATA = {
    "train": lambda: (y_tr, X_tr),
    "val":   lambda: (y_va, X_va),
    "test":  lambda: (y_te, X_te),
}


def _proba_splits(model):
    out = {}
    for split, (y, X) in [(s, _SPLIT_DATA[s]()) for s in ("train", "val", "test")]:
        out[split] = model.predict_proba(X)[:, 1]
    return out


def plot_roc_and_confusion(name, probas, threshold=0.5, op_threshold=None, save=True):
    ys = {s: _SPLIT_DATA[s]()[0] for s in ("train", "val", "test")}
    fig, axes = plt.subplots(2, 3, figsize=(16, 10))
    fig.suptitle(f"{name} — ROC & confusion matrices", fontsize=14, y=1.01)
    for ax, split in zip(axes[0], ("train", "val", "test")):
        y, p = ys[split], probas[split]
        fpr, tpr, _ = roc_curve(y, p)
        auc = roc_auc_score(y, p)
        ax.plot(fpr, tpr, lw=2, label=f"AUC={auc:.3f}")
        ax.plot([0, 1], [0, 1], "k--", alpha=0.35)
        ax.set_title(f"ROC — {split.upper()}")
        ax.legend(loc="lower right")
    thr_map = {"train": threshold, "val": threshold, "test": op_threshold or threshold}
    for ax, split in zip(axes[1], ("train", "val", "test")):
        y, p = ys[split], probas[split]
        thr = thr_map[split]
        pred = (p >= thr).astype(int)
        cm = confusion_matrix(y, pred)
        sns.heatmap(cm, annot=True, fmt=",d", cmap="Blues", ax=ax, cbar=False,
                    xticklabels=["No Fail", "Fail"], yticklabels=["No Fail", "Fail"])
        ax.set_title(f"CM — {split.upper()}  (t={thr:.4f})")
    plt.tight_layout()
    if save:
        safe = "".join(c if c.isalnum() or c in "-_" else "_" for c in name)
        path = os.path.join(OUT_DIR, f"{safe}_roc_confusion.png")
        fig.savefig(path, dpi=120, bbox_inches="tight")
        print(f"  [plot] saved -> {path}")
    plt.show()


def eval_split(y_true, proba, name=""):
    pred05 = (proba >= 0.5).astype(int)
    return {"split": name, "auc": roc_auc_score(y_true, proba),
            "ap": average_precision_score(y_true, proba),
            "acc": (pred05 == y_true).mean(),
            "f1": f1_score(y_true, pred05, zero_division=0),
            "prec": precision_score(y_true, pred05, zero_division=0),
            "rec": recall_score(y_true, pred05, zero_division=0)}


def recall_at_k(y_true, proba, k_pct=0.10):
    k = max(1, int(len(y_true) * k_pct))
    top = np.argsort(proba)[-k:]
    return {"prec_at_k": y_true[top].mean(),
            "recall_at_k": y_true[top].sum() / max(y_true.sum(), 1),
            "lift": y_true[top].mean() / max(y_true.mean(), 1e-9)}


def best_recall_constrained_threshold(y_true, proba, floor=RECALL_FLOOR, max_fleet=0.50):
    """Max precision on val s.t. recall >= floor and fleet <= max_fleet.
    Falls back to best-F1 within fleet cap if recall floor is unreachable."""
    order = np.argsort(proba)[::-1]
    best_t, best_p = None, -1.0
    best_f1_t, best_f1 = None, -1.0
    for i in range(1, len(y_true) + 1):
        if i / len(y_true) > max_fleet:
            break
        t = proba[order[i - 1]]
        pred = (proba >= t).astype(int)
        rec = recall_score(y_true, pred, zero_division=0)
        prec = precision_score(y_true, pred, zero_division=0)
        f1 = f1_score(y_true, pred, zero_division=0)
        if f1 > best_f1:
            best_f1, best_f1_t = f1, t
        if rec >= floor and prec > best_p:
            best_p, best_t = prec, t
    if best_t is not None:
        return best_t, best_p
    return (best_f1_t if best_f1_t is not None else 0.5), max(best_f1, 0.0)


def majority_baseline_acc(y_true):
    """Accuracy if we always predict the majority class (accuracy ceiling for skewed labels)."""
    p = max(y_true.mean(), 1 - y_true.mean())
    return p


def max_acc_threshold(y_true, proba):
    """Threshold that maximizes accuracy (usually predict-all-negative when pos << 50%)."""
    if y_true.mean() <= 0.5:
        return float(np.nextafter(proba.max(), 2.0)), majority_baseline_acc(y_true)
    return float(np.nextafter(proba.min(), -1.0)), majority_baseline_acc(y_true)


def calibrate_isotonic(raw_va, y_va_ref, raw_te):
    iso = IsotonicRegression(out_of_bounds="clip")
    iso.fit(raw_va, y_va_ref)
    cal_va = iso.predict(raw_va)
    cal_te = iso.predict(raw_te)
    if average_precision_score(y_va_ref, cal_va) >= average_precision_score(y_va_ref, raw_va):
        return cal_va, cal_te, iso, True
    return raw_va, raw_te, None, False


def eval_at_threshold(y_true, proba, threshold):
    pred = (proba >= threshold).astype(int)
    return {
        "auc": roc_auc_score(y_true, proba),
        "ap": average_precision_score(y_true, proba),
        "acc": (pred == y_true).mean(),
        "prec": precision_score(y_true, pred, zero_division=0),
        "rec": recall_score(y_true, pred, zero_division=0),
        "f1": f1_score(y_true, pred, zero_division=0),
    }


def val_ap_weights_named(models_dict, X_va_ref, y_va_ref, min_ap=MIN_BOOSTER_VAL_AP):
    """Val AP weights; drop boosters below min_ap (collapsed LGB guard)."""
    active = {}
    for name, m in models_dict.items():
        ap = average_precision_score(y_va_ref, m.predict_proba(X_va_ref)[:, 1])
        if ap >= min_ap:
            active[name] = (m, ap)
        else:
            print(f"  [ensemble] dropping {name}: val_AP={ap:.4f} < {min_ap}")
    if not active:
        raise RuntimeError("All boosters collapsed — check SPW / features")
    names = list(active.keys())
    aps = np.array([active[n][1] for n in names])
    w = aps / aps.sum()
    models = [active[n][0] for n in names]
    return models, w, dict(zip(names, aps.round(4))), names


def print_model_report(name, model=None, probas=None):
    if probas is None:
        probas = _proba_splits(model)
    rows = [eval_split(_SPLIT_DATA[s]()[0], probas[s], s) for s in ("train", "val", "test")]
    tab = pd.DataFrame(rows)
    print(f"\n{name}")
    print(tab.to_string(index=False, float_format=lambda x: f"{x:.4f}"))
    t_rc, p_rc = best_recall_constrained_threshold(y_va, probas["val"])
    pred_te = (probas["test"] >= t_rc).astype(int)
    rk = recall_at_k(y_te, probas["test"], 0.10)
    print(f"  Val threshold (rec>={RECALL_FLOOR}): {t_rc:.4f}  val_prec={p_rc:.4f}")
    print(f"  Test @ threshold: prec={precision_score(y_te,pred_te,zero_division=0):.4f}  "
          f"rec={recall_score(y_te,pred_te,zero_division=0):.4f}  fleet={(pred_te==1).mean():.1%}")
    print(f"  Test @ top-10%: prec={rk['prec_at_k']:.4f}  rec={rk['recall_at_k']:.4f}  lift={rk['lift']:.2f}x")
    plot_roc_and_confusion(name, probas, threshold=0.5, op_threshold=t_rc)
    return tab, probas["test"]

In [ ]:
# CELL 9 — Logistic Regression baseline
lr = Pipeline([
    ("sc", StandardScaler()),
    ("clf", LogisticRegression(C=0.05, class_weight="balanced", max_iter=2000, solver="lbfgs", random_state=42)),
])
lr.fit(X_tr_fit, y_tr_fit)
lr_tab, _ = print_model_report("lr_improved", lr)

In [ ]:
# CELL 9b — Optional Optuna HPT (LGB / XGB / CB) — set USE_OPTUNA=True
import joblib as _joblib_opt

if not USE_OPTUNA:
    print("[Optuna] USE_OPTUNA=False — skip; Cells 10-12 train default hyperparams")
else:
    import optuna
    from sklearn.model_selection import TimeSeriesSplit

    optuna.logging.set_verbosity(optuna.logging.WARNING)
    _MODEL_TAG = "optuna"
    X_cv = np.vstack([X_tr, X_va])
    y_cv = np.concatenate([y_tr, y_va])
    TS_CV = TimeSeriesSplit(n_splits=3)

    if OPTUNA_TUNE_ROWS and len(X_cv) > OPTUNA_TUNE_ROWS:
        _tune_idx = np.sort(np.random.default_rng(42).choice(
            len(X_cv), size=OPTUNA_TUNE_ROWS, replace=False))
        _X_tune, _y_tune = X_cv[_tune_idx], y_cv[_tune_idx]
        print(f"[Optuna] Search sample: {OPTUNA_TUNE_ROWS:,}/{len(X_cv):,} rows")
    else:
        _X_tune, _y_tune = X_cv, y_cv

    _saved = {}
    if not FORCE_OPTUNA and os.path.exists(OPTUNA_CKPT):
        _saved = _joblib_opt.load(OPTUNA_CKPT)
        print(f"[Optuna] Checkpoint: {list(_saved.keys())}")

    def _trial_log(study, trial):
        if trial.state == optuna.trial.TrialState.COMPLETE:
            flag = "  *** BEST" if trial.value == study.best_value else ""
            print(f"    trial {trial.number:3d}  AP={trial.value:.4f}  best={study.best_value:.4f}{flag}")

    if "xgb_tvm" in _saved and not FORCE_OPTUNA:
        _xgb_p = {**_saved["xgb_tvm"]["best_params"], "scale_pos_weight": spw_fit,
                  "eval_metric": "aucpr", "early_stopping_rounds": 40,
                  "tree_method": "hist", "random_state": 42, "n_jobs": -1}
        _xgb_cv_ap = _saved["xgb_tvm"]["best_cv_ap"]
        print(f"\n[XGB] checkpoint CV_AP={_xgb_cv_ap:.4f}")
    else:
        def _xgb_obj(trial):
            p = {"n_estimators": trial.suggest_int("n_estimators", 200, 800),
                 "max_depth": trial.suggest_int("max_depth", 4, 9),
                 "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
                 "subsample": trial.suggest_float("subsample", 0.5, 1.0),
                 "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
                 "min_child_weight": trial.suggest_int("min_child_weight", 1, 30),
                 "gamma": trial.suggest_float("gamma", 0.0, 3.0),
                 "reg_alpha": trial.suggest_float("reg_alpha", 1e-4, 10.0, log=True),
                 "reg_lambda": trial.suggest_float("reg_lambda", 1e-4, 10.0, log=True),
                 "scale_pos_weight": spw_fit, "eval_metric": "aucpr",
                 "early_stopping_rounds": 30, "tree_method": "hist",
                 "random_state": 42, "n_jobs": -1}
            scores = []
            for step, (tr, va) in enumerate(TS_CV.split(_X_tune)):
                m = XGBClassifier(**p)
                m.fit(_X_tune[tr], _y_tune[tr], eval_set=[(_X_tune[va], _y_tune[va])], verbose=False)
                scores.append(average_precision_score(_y_tune[va], m.predict_proba(_X_tune[va])[:, 1]))
                trial.report(np.mean(scores), step)
                if trial.should_prune(): raise optuna.TrialPruned()
            return float(np.mean(scores))
        print(f"\n[XGB] TPE n={OPTUNA_TRIALS['xgb']} timeout={OPTUNA_TIMEOUT_SEC['xgb']}s")
        _xs = optuna.create_study(direction="maximize",
            sampler=optuna.samplers.TPESampler(seed=42),
            pruner=optuna.pruners.MedianPruner(n_startup_trials=8, n_warmup_steps=1))
        _xs.optimize(_xgb_obj, n_trials=OPTUNA_TRIALS["xgb"],
                     timeout=OPTUNA_TIMEOUT_SEC["xgb"], callbacks=[_trial_log])
        _xgb_cv_ap = _xs.best_value
        _xgb_p = {**_xs.best_params, "scale_pos_weight": spw_fit, "eval_metric": "aucpr",
                  "early_stopping_rounds": 40, "tree_method": "hist", "random_state": 42, "n_jobs": -1}
        _saved["xgb_tvm"] = {"best_params": _xs.best_params, "best_cv_ap": _xgb_cv_ap}
        _joblib_opt.dump(_saved, OPTUNA_CKPT)
    xgb = XGBClassifier(**_xgb_p)
    xgb.fit(X_tr_fit, y_tr_fit, sample_weight=_w_fit, eval_set=[(X_va, y_va)], verbose=False)
    xgb_tab, xgb_proba = print_model_report("xgb_optuna", xgb)

    if "lgb_tvm" in _saved and not FORCE_OPTUNA:
        _lgb_p = {**_saved["lgb_tvm"]["best_params"], "scale_pos_weight": spw_fit,
                  "random_state": 42, "n_jobs": -1, "verbose": -1}
        _lgb_cv_ap = _saved["lgb_tvm"]["best_cv_ap"]
        print(f"\n[LGB] checkpoint CV_AP={_lgb_cv_ap:.4f}")
    else:
        def _lgb_obj(trial):
            p = {"n_estimators": trial.suggest_int("n_estimators", 200, 800),
                 "max_depth": trial.suggest_int("max_depth", 4, 10),
                 "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
                 "num_leaves": trial.suggest_int("num_leaves", 20, 200),
                 "subsample": trial.suggest_float("subsample", 0.5, 1.0),
                 "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
                 "min_child_samples": trial.suggest_int("min_child_samples", 5, 100),
                 "reg_alpha": trial.suggest_float("reg_alpha", 1e-4, 10.0, log=True),
                 "reg_lambda": trial.suggest_float("reg_lambda", 1e-4, 10.0, log=True),
                 "scale_pos_weight": spw_fit, "random_state": 42, "n_jobs": -1, "verbose": -1}
            scores = []
            for step, (tr, va) in enumerate(TS_CV.split(_X_tune)):
                m = LGBMClassifier(**p)
                m.fit(_X_tune[tr], _y_tune[tr], eval_set=[(_X_tune[va], _y_tune[va])],
                      callbacks=[early_stopping(30, verbose=False), log_evaluation(0)])
                scores.append(average_precision_score(_y_tune[va], m.predict_proba(_X_tune[va])[:, 1]))
                trial.report(np.mean(scores), step)
                if trial.should_prune(): raise optuna.TrialPruned()
            return float(np.mean(scores))
        print(f"\n[LGB] TPE n={OPTUNA_TRIALS['lgb']} timeout={OPTUNA_TIMEOUT_SEC['lgb']}s")
        _ls = optuna.create_study(direction="maximize",
            sampler=optuna.samplers.TPESampler(seed=42),
            pruner=optuna.pruners.MedianPruner(n_startup_trials=6, n_warmup_steps=1))
        _ls.optimize(_lgb_obj, n_trials=OPTUNA_TRIALS["lgb"],
                     timeout=OPTUNA_TIMEOUT_SEC["lgb"], callbacks=[_trial_log])
        _lgb_cv_ap = _ls.best_value
        _lgb_p = {**_ls.best_params, "scale_pos_weight": spw_fit,
                  "random_state": 42, "n_jobs": -1, "verbose": -1}
        _saved["lgb_tvm"] = {"best_params": _ls.best_params, "best_cv_ap": _lgb_cv_ap}
        _joblib_opt.dump(_saved, OPTUNA_CKPT)
    lgb = LGBMClassifier(**_lgb_p)
    lgb.fit(X_tr_fit, y_tr_fit, sample_weight=_w_fit, eval_set=[(X_va, y_va)],
            callbacks=[early_stopping(40, verbose=False), log_evaluation(0)])
    lgb_tab, lgb_proba = print_model_report("lgb_optuna", lgb)

    if "cb_tvm" in _saved and not FORCE_OPTUNA:
        _cb_p = {**_saved["cb_tvm"]["best_params"], "scale_pos_weight": spw_fit,
                 "random_seed": 42, "verbose": 0}
        _cb_cv_ap = _saved["cb_tvm"]["best_cv_ap"]
        print(f"\n[CB] checkpoint CV_AP={_cb_cv_ap:.4f}")
    else:
        def _cb_obj(trial):
            p = {"iterations": trial.suggest_int("iterations", 200, 800),
                 "depth": trial.suggest_int("depth", 4, 9),
                 "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
                 "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1.0, 20.0, log=True),
                 "border_count": trial.suggest_int("border_count", 32, 255),
                 "bagging_temperature": trial.suggest_float("bagging_temperature", 0.0, 2.0),
                 "random_strength": trial.suggest_float("random_strength", 0.5, 3.0),
                 "scale_pos_weight": spw_fit, "random_seed": 42, "verbose": 0}
            scores = []
            for step, (tr, va) in enumerate(TS_CV.split(_X_tune)):
                m = CatBoostClassifier(**p)
                m.fit(_X_tune[tr], _y_tune[tr], eval_set=(_X_tune[va], _y_tune[va]), early_stopping_rounds=30)
                scores.append(average_precision_score(_y_tune[va], m.predict_proba(_X_tune[va])[:, 1]))
                trial.report(np.mean(scores), step)
                if trial.should_prune(): raise optuna.TrialPruned()
            return float(np.mean(scores))
        print(f"\n[CB] TPE n={OPTUNA_TRIALS['cb']} timeout={OPTUNA_TIMEOUT_SEC['cb']}s")
        _cs = optuna.create_study(direction="maximize",
            sampler=optuna.samplers.TPESampler(seed=42),
            pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=1))
        _cs.optimize(_cb_obj, n_trials=OPTUNA_TRIALS["cb"],
                     timeout=OPTUNA_TIMEOUT_SEC["cb"], callbacks=[_trial_log])
        _cb_cv_ap = _cs.best_value
        _cb_p = {**_cs.best_params, "scale_pos_weight": spw_fit, "random_seed": 42, "verbose": 0}
        _saved["cb_tvm"] = {"best_params": _cs.best_params, "best_cv_ap": _cb_cv_ap}
        _joblib_opt.dump(_saved, OPTUNA_CKPT)
    cb = CatBoostClassifier(**_cb_p)
    cb.fit(X_tr_fit, y_tr_fit, sample_weight=_w_fit, eval_set=(X_va, y_va), use_best_model=True)
    cb_tab, cb_proba = print_model_report("cb_optuna", cb)
    print(f"\n[Optuna] Done — checkpoint: {OPTUNA_CKPT}")

In [ ]:
# CELL 10 — LightGBM (prod-aligned: natural SPW, recency weights, 600 trees)
if USE_OPTUNA:
    print("[Cell 10] skipped — lgb trained in Cell 9b (lgb_optuna)")
else:
    lgb = LGBMClassifier(
        n_estimators=600, learning_rate=0.05, num_leaves=63, max_depth=-1,
        min_child_samples=20, subsample=0.8, colsample_bytree=0.8,
        reg_alpha=0.1, reg_lambda=1.0, scale_pos_weight=spw_fit,
        random_state=42, n_jobs=-1, verbose=-1, metric="auc",
    )
    lgb.fit(X_tr_fit, y_tr_fit, sample_weight=_w_fit, eval_set=[(X_va, y_va)],
            callbacks=[early_stopping(40, verbose=False), log_evaluation(0)])
    lgb_tab, lgb_proba = print_model_report("lgb_improved", lgb)

In [ ]:
# CELL 11 — XGBoost (prod-aligned)
if USE_OPTUNA:
    print("[Cell 11] skipped — xgb trained in Cell 9b (xgb_optuna)")
else:
    xgb = XGBClassifier(
        n_estimators=600, learning_rate=0.05, max_depth=6, min_child_weight=10,
        subsample=0.8, colsample_bytree=0.8, reg_alpha=0.1, reg_lambda=1.0,
        scale_pos_weight=spw_fit, eval_metric="auc", random_state=42, n_jobs=-1,
        early_stopping_rounds=40, tree_method="hist",
    )
    xgb.fit(X_tr_fit, y_tr_fit, sample_weight=_w_fit, eval_set=[(X_va, y_va)], verbose=False)
    xgb_tab, xgb_proba = print_model_report("xgb_improved", xgb)

In [ ]:
# CELL 12 — CatBoost + RF (prod uses RF in stacking; strong TVM baseline)
from sklearn.ensemble import RandomForestClassifier

if USE_OPTUNA:
    print("[Cell 12] cb trained in Cell 9b — training RF only")
else:
    cb = CatBoostClassifier(
        iterations=600, learning_rate=0.05, depth=7, l2_leaf_reg=3,
        scale_pos_weight=spw_fit, random_seed=42, verbose=0,
        eval_metric="AUC", od_type="Iter", od_wait=40,
    )
    cb.fit(X_tr_fit, y_tr_fit, sample_weight=_w_fit, eval_set=(X_va, y_va), use_best_model=True)
    cb_tab, cb_proba = print_model_report("cb_improved", cb)

rf = RandomForestClassifier(
    n_estimators=300, max_depth=12, min_samples_leaf=10,
    class_weight="balanced_subsample", n_jobs=-1, random_state=42,
)
rf.fit(X_tr_fit, y_tr_fit, sample_weight=_w_fit)
rf_tab, rf_proba = print_model_report("rf_improved", rf)

In [ ]:
# CELL 13 — Ensemble + deployment candidate (LR + boosters, prod-style isotonic)
_boosters_all = {"lgb": lgb, "xgb": xgb, "cb": cb, "rf": rf}
_active_models, _ens_w, _va_aps, _ens_names = val_ap_weights_named(_boosters_all, X_va, y_va)
print(f"Booster ensemble val AP: {_va_aps}")
print(f"  weights={dict(zip(_ens_names, _ens_w.round(3)))}")

def _weighted_proba(models, weights, X):
    P = np.column_stack([m.predict_proba(X)[:, 1] for m in models])
    return P @ weights

p_tr_ens = _weighted_proba(_active_models, _ens_w, X_tr)
p_va_ens = _weighted_proba(_active_models, _ens_w, X_va)
p_te_ens = _weighted_proba(_active_models, _ens_w, X_te)

ens_probas = {"train": p_tr_ens, "val": p_va_ens, "test": p_te_ens}
ens_tab, _ = print_model_report("weighted_ensemble", probas=ens_probas)

# Deploy pool: LR + active boosters (val AP weights)
_deploy_dict = {"lr": lr, **dict(zip(_ens_names, _active_models))}
_deploy_models, _deploy_w, _deploy_va_aps, _deploy_names = val_ap_weights_named(_deploy_dict, X_va, y_va)
print(f"Deploy val AP: {_deploy_va_aps}")
print(f"  deploy weights={dict(zip(_deploy_names, _deploy_w.round(3)))}")

p_va_deploy_raw = _weighted_proba(_deploy_models, _deploy_w, X_va)
p_te_deploy_raw = _weighted_proba(_deploy_models, _deploy_w, X_te)

class DeployAPEnsemble:
    """Val AP-weighted LR + LGB/XGB/CB/RF + optional isotonic — SageMaker deploy artifact."""
    _estimator_type = "classifier"
    def __init__(self, models, weights, iso=None):
        self.models = models
        self.weights = np.asarray(weights)
        self.iso = iso
    def predict_proba(self, X):
        P = np.column_stack([m.predict_proba(X)[:, 1] for m in self.models])
        raw = (P * self.weights).sum(axis=1)
        if self.iso is not None:
            cal = self.iso.predict(raw)
            return np.column_stack([1 - cal, cal])
        return np.column_stack([1 - raw, raw])
    def predict(self, X):
        return (self.predict_proba(X)[:, 1] >= 0.5).astype(int)

_deploy_iso = None
if USE_ISOTONIC:
    _cal_va, _cal_te, _deploy_iso, _iso_ok = calibrate_isotonic(p_va_deploy_raw, y_va, p_te_deploy_raw)
    _deploy_tag = "deploy_ap_isotonic_ensemble" if _iso_ok else "deploy_ap_ensemble (isotonic skipped)"
else:
    _cal_va, _cal_te, _deploy_iso, _iso_ok = p_va_deploy_raw, p_te_deploy_raw, None, False
    _deploy_tag = "deploy_ap_ensemble"

deploy_model = DeployAPEnsemble(_deploy_models, _deploy_w, _deploy_iso)
deploy_probas = {
    "train": deploy_model.predict_proba(X_tr)[:, 1],
    "val": _cal_va,
    "test": _cal_te,
}
deploy_tab, deploy_proba = print_model_report(_deploy_tag, probas=deploy_probas)
print(f"  [deploy] champion candidate: {_deploy_tag}  iso={'yes' if _iso_ok else 'no'}")

In [ ]:
# CELL 14 — Leaderboard (AP primary) + accuracy context + deploy candidate
from sklearn.metrics import balanced_accuracy_score

leaderboard = pd.concat([
    lr_tab.assign(model="lr_improved"),
    lgb_tab.assign(model=f"lgb_{_MODEL_TAG}"),
    xgb_tab.assign(model=f"xgb_{_MODEL_TAG}"),
    cb_tab.assign(model=f"cb_{_MODEL_TAG}"),
    rf_tab.assign(model="rf_improved"),
    ens_tab.assign(model="weighted_ensemble"),
    deploy_tab.assign(model=_deploy_tag),
])
test_lb = leaderboard[leaderboard["split"] == "test"].sort_values("ap", ascending=False)

_op_t, _op_val_prec = best_recall_constrained_threshold(y_va, deploy_probas["val"])
_op_row = eval_at_threshold(y_te, deploy_proba, _op_t)
_op_row["model"] = f"{_deploy_tag} @op_t"
_op_row["split"] = "test"

_acc_t, _max_acc = max_acc_threshold(y_te, deploy_proba)
_max_acc_row = eval_at_threshold(y_te, deploy_proba, _acc_t)
_max_acc_row["model"] = f"{_deploy_tag} @max_acc (predict-mostly-negative)"
_max_acc_row["split"] = "test"

_maj_acc = majority_baseline_acc(y_te)
_test_pos = y_te.mean()
_deploy_bal = balanced_accuracy_score(y_te, (deploy_proba >= 0.5).astype(int))

print("\n=== LABEL / ACCURACY CONTEXT (test) ===")
print(f"  positive rate={_test_pos:.2%}  majority-class acc ceiling={_maj_acc:.2%}")
print(f"  -> 90% accuracy is NOT reachable on this split unless pos rate < 10%")
print(f"  -> use AP + Recall@10% + @op_t for ops; acc@0.5 penalizes failure recall")
print(f"  deploy @0.5 balanced_acc={_deploy_bal:.4f}  (better for imbalanced ranking)")

print("\n=== TEST LEADERBOARD (by AP, threshold=0.5) ===")
print(test_lb[["model", "auc", "ap", "acc", "prec", "rec", "f1"]].to_string(
    index=False, float_format=lambda x: f"{x:.4f}"))

print(f"\n=== DEPLOY @ operational threshold (t={_op_t:.4f} from val, rec>={RECALL_FLOOR}) ===")
print(pd.DataFrame([_op_row])[["model", "auc", "ap", "acc", "prec", "rec", "f1"]].to_string(
    index=False, float_format=lambda x: f"{x:.4f}"))

print(f"\n=== DEPLOY @ max-accuracy threshold (t={_acc_t:.4f}, acc ceiling ~{_maj_acc:.2%}) ===")
print(pd.DataFrame([_max_acc_row])[["model", "auc", "ap", "acc", "prec", "rec", "f1"]].to_string(
    index=False, float_format=lambda x: f"{x:.4f}"))

prod = pd.DataFrame([
    {"model": "lgb_tvm (prod)",        "auc": 0.6448, "ap": 0.3385, "acc": np.nan},
    {"model": "cb_optuna_tvm (prod)",  "auc": 0.6449, "ap": 0.3379, "acc": np.nan},
    {"model": "xgb_optuna_tvm (prod)", "auc": 0.6404, "ap": 0.3356, "acc": np.nan},
])
print("\n=== Production reference (AP — prod does not optimize accuracy) ===")
print(prod.to_string(index=False))
best = test_lb.iloc[0]
_delta = best["ap"] - 0.3385
rk = recall_at_k(y_te, deploy_proba, 0.10)
print(f"\nBest @0.5: {best['model']}  AUC={best['auc']:.4f}  AP={best['ap']:.4f}  acc={best['acc']:.4f}  (delta vs prod AP: {_delta:+.4f})")
print(f"Deploy @ top-10%: prec={rk['prec_at_k']:.4f}  rec={rk['recall_at_k']:.4f}  lift={rk['lift']:.2f}x")
print("\n=== To close AP gap vs prod — enable Optuna or try ===")
print("  1. USE_OPTUNA=True in Cell 2, re-run Cells 9b-15 (~15-25 min first run)")
print("  2. Honest stacking + Platt calib (prod Cell 28)")
print("  3. MI_TOP_N=150 + verify subsystem roll7 / sales_7d lags post-prune")
print("  4. Guarded SPW sweep (USE_SPW_SWEEP=True) — only if val AP improves")

In [ ]:
# CELL 15 — All models: test-set ROC overlay
_all_models = {
    "lr_improved": lr.predict_proba(X_te)[:, 1],
    f"lgb_{_MODEL_TAG}": lgb_proba,
    f"xgb_{_MODEL_TAG}": xgb_proba,
    f"cb_{_MODEL_TAG}": cb_proba,
    "rf_improved": rf_proba,
    "weighted_ensemble": p_te_ens,
    _deploy_tag: deploy_proba,
}

fig, ax = plt.subplots(figsize=(8, 6))
for name, proba in _all_models.items():
    fpr, tpr, _ = roc_curve(y_te, proba)
    ax.plot(fpr, tpr, lw=2, label=f"{name}  AUC={roc_auc_score(y_te, proba):.3f}")
ax.plot([0, 1], [0, 1], "k--", alpha=0.35)
ax.set_title("TEST ROC — all TVM models")
ax.set_xlabel("False positive rate")
ax.set_ylabel("True positive rate")
ax.legend(loc="lower right", fontsize=9)
plt.tight_layout()
path = os.path.join(OUT_DIR, "all_models_test_roc.png")
fig.savefig(path, dpi=120, bbox_inches="tight")
print(f"[plot] saved -> {path}")
plt.show()